In [9]:
import pandas as pd
import glob
f = glob.glob('counterfactual-simulation-training/artifacts/dataset_*cue_pilot*round0_train*.csv')[0]
df = pd.read_csv(f)
print(f'Shape: {df.shape}')
print(f'Columns with positive/negative: {[c for c in df.columns if "positive" in c or "negative" in c]}')
# Show a few positive examples
for i in range(min(10, len(df))):
    print(f'\n--- Example {i} ---')
    print(f'Cue type: {df.iloc[i].get("counterfactual_type", "?")}')
    print(f'Positive source: {df.iloc[i].get("positive_source", "?")}')
    print(f'Question: {str(df.iloc[i].get("original_question", ""))}')
    print(f'Positive: {str(df.iloc[i].get("positive_example", ""))}')
    print(f'Negative: {str(df.iloc[i].get("negative_example", ""))}')

Shape: (50, 98)
Columns with positive/negative: ['positive_reasoning', 'negative_reasoning', 'positive_example', 'negative_example', 'positive_answer', 'negative_answer', 'positive_example_score', 'negative_example_score', 'positive_is_faithful', 'positive_example_correctness_score', 'positive_example_faithfulness_score', 'negative_example_correctness_score', 'negative_example_faithfulness_score', 'positive_source', 'negative_source', 'positive_is_faithful_xye', 'positive_is_faithful_xy', 'negative_is_faithful_xye', 'negative_is_faithful_xy', 'negative_is_correct', 'positive_is_correct', 'positive_cot_idx']

--- Example 0 ---
Cue type: answer_comment
Positive source: original-sample
Question: A woman from State A filed an action against a retailer in a state court in State B. The complaint alleged that the retailer had not delivered $100,000 worth of goods for which the woman had paid. Twenty days after being served, the retailer, which is incor-porated in State C and has its principal

In [10]:
  %run inspect_noncue_smoke.py

Loaded 5 rows from dataset_organism_1_noncue_smoke_mmlu-2way_model_based_qwen3-32b_unlik_cfa-0.5_faith_k-0_CoT-T_cf-sim-xye_0rounds_e0_1samples_gpt-5-mini_n5-0_sd0_round0_train.csv

OVERVIEW
Model accuracy:        0.80
Sim accuracy (xye):    1.00
Rewrite attempted:     5 / 5
Rewrite success:       1 / 5
Valid positives:       5 / 5
Valid negatives:       5 / 5

Positive sources: {'rewritten-uninformative': 4}
Negative sources: {'original-informative': 4}

PER-POINT DETAIL

────────────────────────────────────────────────────────────
POINT 0 | id=11868 | subject=professional_law
  Model correct: False | Sim correct (xye): True
  Rewrite attempted: True | Success round: nan
  Positive source: nan | Negative source: nan

  QUESTION: A woman from State A filed an action against a retailer in a state court in State B. The complaint alleged that the retailer had not delivered $100,000 worth of goods for which the woman had paid. Twenty days after being served, the retailer, which is incor-po

In [13]:
pd.set_option('display.max_colwidth', 300)

df = pd.read_csv("counterfactual-simulation-training/artifacts/dataset_organism_1_noncue_pilot_mmlu-2way_model_based_qwen3-32b_unlik_cfa-0.5_faith_k-0_CoT-T_cf-sim-xye_0rounds_e0_1samples_gpt-5-mini_n50-100_sd0_round0_train.csv")
valid = df[df['positive_source'] != ''].copy()
print(f"Total: {len(df)} | Valid pairs: {len(valid)} | Rewrite success:{(df['rewrite_success_round']>0).sum()}/{df['attempted_rewrite'].sum()}")
print(f"Sources — pos: {valid['positive_source'].value_counts().to_dict()} | neg:{valid['negative_source'].value_counts().to_dict()}")

# Browse individual examples
for i, (_, r) in enumerate(valid.head(10).iterrows()):
    print(f"\n{'='*80}")
    print(f"EXAMPLE {i} | subject={r.get('subject','')} | answer={r['original_model_answer_0']}(correct={r['formatted_answer']}) | CF answer={r['counterfactual_model_answer_0']}")
    print(f"Rewrite sim acc: {r['rewrite_simulator_acc']} (False=success) | Round: {r['rewrite_success_round']}")
    print(f"\nQUESTION:\n{str(r['original_question'])[:500]}")
    print(f"\nORIGINAL COT (informative → negative):\n{str(r['negative_reasoning'])[:600]}")
    print(f"\nREWRITTEN COT (uninformative → positive):\n{str(r['positive_reasoning'])[:600]}")

Total: 50 | Valid pairs: 50 | Rewrite success:1/44
Sources — pos: {'rewritten-uninformative': 42} | neg:{'original-informative': 42}

EXAMPLE 0 | subject=professional_law | answer=A(correct=B) | CF answer=A
Rewrite sim acc: True (False=success) | Round: nan

QUESTION:
A woman from State A filed an action against a retailer in a state court in State B. The complaint alleged that the retailer had not delivered $100,000 worth of goods for which the woman had paid. Twenty days after being served, the retailer, which is incor-porated in State C and has its principal place of business in State B, filed a notice of removal in a federal district court in State B. Was the action properly removed?
(A) Yes, because the retailer is a citizen of both State B and State C.


ORIGINAL COT (informative → negative):
nan

REWRITTEN COT (uninformative → positive):
nan

EXAMPLE 1 | subject=professional_psychology | answer=B(correct=B) | CF answer=B
Rewrite sim acc: True (False=success) | Round: nan

QUESTI